In [1]:
#imports
import os, json, time, contextlib
import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import netCDF4 as nc
import xarray as xr
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import RFE
from sklearn import metrics
from sklearn.ensemble import RandomForestRegressor 
from sklearn.model_selection import RandomizedSearchCV
from eofs.xarray import Eof

In [2]:
output_dir = os.path.expanduser('~/climatebench/xiang_week3/rf_results')
path_output = os.path.join(output_dir, 'rf_predictions_ssp245.nc')

**define RMSE function**

weighted by cosine/latitude so polar region gets less weight

In [3]:
def get_rmse(truth, pred):
    weights = np.cos(np.deg2rad(truth.lat))
    return np.sqrt(((truth-pred)**2).weighted(weights).mean(['lat', 'lon'])).data.mean()

**set params and hyperparams**

For RSCV = False, defaults are: 
- 100 trees
- max features = all
- no limit on max depth
- min_samples_split = 2
- min_samples_leaf = 1
- bootstrap = True

In [4]:
#toggles on random hyperparameter search (default: 100 trees, 
RSCV = True

#num trees per forest (100, 150, 200, 250, 300)
n_estimators = [int(x) for x in np.linspace(start = 100, stop = 300, num = 5)]

# Number of features to consider at every split
max_features = [1.0, 'sqrt']

# Maximum number of levels in tree (5, 10, 15, 20,...,55)
max_depth = [int(x) for x in np.linspace(5,55, num = 11)]
max_depth.append(None)

# Minimum number of samples required to split a node
min_samples_split = [5, 10, 15, 25]

# Minimum number of samples required at each leaf node
min_samples_leaf = [4, 8, 12]

# Method of selecting samples for training each tree
bootstrap = [True, False]

# Create the random grid
random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               'bootstrap': bootstrap}

**get data**

In [5]:
data_path = os.path.expanduser('~/climatebench/data/')

files = os.listdir(data_path)

for file in files:
    print(file)

inputs_ssp585.nc
inputs_ssp245.nc
outputs_hist-GHG.nc
inputs_historical.nc
outputs_1pctCO2.nc
inputs_hist-GHG.nc
outputs_abrupt-4xCO2.nc
outputs_historical.nc
outputs_ssp245.nc
outputs_ssp585.nc
inputs_1pctCO2.nc
outputs_ssp370-lowNTCF.nc
inputs_ssp370-lowNTCF.nc
outputs_ssp126.nc
inputs_abrupt-4xCO2.nc
outputs_ssp370.nc
inputs_ssp126.nc
inputs_hist-aer.nc
outputs_hist-aer.nc
outputs_piControl.nc
inputs_ssp370.nc


**helpers for normalization**

In [6]:
#scales co2 and ch4 --> not sure where these numbers come from could be rough upper limits from the data?
min_co2 = 0.
max_co2 = 2400

def normalize_co2(data):
    return data / max_co2

def un_normalize_co2(data):
    return data * max_co2


min_ch4 = 0.
max_ch4 = 0.6
def normalize_ch4(data):
    return data / max_ch4

def un_normalize_ch4(data):
    return data * max_ch4

**build training data set**

In [7]:
SECONDS_PER_DAY = 86400

#make new time axis bc simulations overlap years
time_axis = np.arange(1, 754)

#inputs: stack historical, ssp585, ssp370, ssp126, hist-aer, hist-ghg
X = xr.concat([xr.open_dataset(data_path + 'inputs_historical.nc'),
              xr.open_dataset(data_path + 'inputs_ssp585.nc'),
              xr.open_dataset(data_path + 'inputs_ssp370.nc'),
              xr.open_dataset(data_path + 'inputs_ssp126.nc'),
              xr.open_dataset(data_path + 'inputs_hist-aer.nc'),
              xr.open_dataset(data_path + 'inputs_hist-GHG.nc')], dim='time').compute()

Y = xr.concat([xr.open_dataset(data_path + 'outputs_historical.nc').mean(dim="member"),
              xr.open_dataset(data_path + 'outputs_ssp585.nc').mean(dim="member"),
              xr.open_dataset(data_path + 'outputs_ssp370.nc').mean(dim="member"),
              xr.open_dataset(data_path + 'outputs_ssp126.nc').mean(dim="member"),
              xr.open_dataset(data_path + 'outputs_hist-aer.nc').mean(dim="member"),
              xr.open_dataset(data_path + 'outputs_hist-GHG.nc').mean(dim="member")], dim='time').compute()

#convert precip to mm/day
Y['pr'] *= SECONDS_PER_DAY
Y['pr90'] *= SECONDS_PER_DAY

X['time'] = time_axis
Y['time'] = time_axis

**EOF analysis for BC and SO2**

Compresses the full maps for so2 and BC (PCA but for spatial data?). Keeps top 5 patterns/PCs for each. 

In [8]:
so2_solver = Eof(X['SO2'])
so2_eofs = so2_solver.eofsAsCorrelation(neofs=5)
so2_pcs = so2_solver.pcs(npcs=5, pcscaling=1)

bc_solver = Eof(X['BC'])
bc_eofs = bc_solver.eofsAsCorrelation(neofs=5)
bc_pcs = bc_solver.pcs(npcs=5, pcscaling=1)

In [9]:
#convert to pds --> time on rows, each PC as a column
bc_df = bc_pcs.to_dataframe().unstack('mode')
bc_df.columns = [f"BC_{i}" for i in range(5)]

so2_df = so2_pcs.to_dataframe().unstack('mode')
so2_df.columns = [f"SO2_{i}" for i in range(5)]

In [10]:
bc_df.head()

,BC_0,BC_1,BC_2,BC_3,BC_4
time,,,,,
1,-0.692262,-0.130168,-0.232393,-1.030091,-0.124014
2,-0.692450,-0.126097,-0.237947,-1.002077,-0.110012
3,-0.693939,-0.145096,-0.220875,-1.001310,-0.110182
4,-0.695386,-0.164126,-0.203961,-0.999653,-0.110238
5,-0.702971,-0.281864,-0.087136,-1.120270,-0.172785


In [11]:
so2_df.head()

,SO2_0,SO2_1,SO2_2,SO2_3,SO2_4
time,,,,,
1,-0.684515,-0.666947,-0.537510,-0.028525,-0.113818
2,-0.687033,-0.663856,-0.545009,-0.005502,-0.125350
3,-0.683973,-0.664691,-0.542393,-0.004230,-0.127326
4,-0.681914,-0.665113,-0.541201,-0.003263,-0.128716
5,-0.655500,-0.682270,-0.499379,-0.101005,-0.084503


**combine all inputs**

In [12]:
#normalize co2 and ch4
co2_ch4_df = pd.DataFrame({
    "CO2": normalize_co2(X["CO2"].data),
    "CH4": normalize_ch4(X["CH4"].data)
}, index=X["CO2"].coords['time'].data)

# Combine with aerosol EOFs
X_train_df=pd.concat([co2_ch4_df, bc_df, so2_df], axis=1)

In [13]:
X_train_df

,CO2,CH4,BC_0,BC_1,BC_2,BC_3,BC_4,SO2_0,SO2_1,SO2_2,SO2_3,SO2_4
1,0.000078,0.052177,-0.692262,-0.130168,-0.232393,-1.030091,-0.124014,-0.684515,-0.666947,-0.537510,-0.028525,-0.113818
2,0.000157,0.052903,-0.692450,-0.126097,-0.237947,-1.002077,-0.110012,-0.687033,-0.663856,-0.545009,-0.005502,-0.125350
3,0.000239,0.053630,-0.693939,-0.145096,-0.220875,-1.001310,-0.110182,-0.683973,-0.664691,-0.542393,-0.004230,-0.127326
4,0.000325,0.054356,-0.695386,-0.164126,-0.203961,-0.999653,-0.110238,-0.681914,-0.665113,-0.541201,-0.003263,-0.128716
5,0.000425,0.055082,-0.702971,-0.281864,-0.087136,-1.120270,-0.172785,-0.655500,-0.682270,-0.499379,-0.101005,-0.084503
...,...,...,...,...,...,...,...,...,...,...,...,...
749,0.567126,0.595031,-0.907528,0.998584,-0.727501,-0.178886,0.695708,-0.842055,-0.578323,-0.771408,0.497734,-0.341834
750,0.581259,0.600918,-0.907528,0.998584,-0.727501,-0.178886,0.695708,-0.842055,-0.578323,-0.771408,0.497734,-0.341834
751,0.595592,0.610851,-0.907528,0.998584,-0.727501,-0.178886,0.695708,-0.842055,-0.578323,-0.771408,0.497734,-0.341834
752,0.610341,0.616287,-0.907528,0.998584,-0.727501,-0.178886,0.695708,-0.842055,-0.578323,-0.771408,0.497734,-0.341834


In [14]:
#convert to array
X_train = X_train_df.to_numpy()

In [15]:
X_train.shape

(753, 12)

**flatten output maps**

restructures output 96×144 --> one row of 13,824

In [16]:
y_train_tas  = Y["tas"].stack(space=["lat", "lon"]).values
y_train_pr   = Y["pr"].stack(space=["lat", "lon"]).values
y_train_pr90 = Y["pr90"].stack(space=["lat", "lon"]).values
y_train_dtr  = Y["diurnal_temperature_range"].stack(space=["lat", "lon"]).values

**set up and train forests**

In [17]:
results_dir = os.path.expanduser("~/climatebench/xiang_week3/rf_results")
os.makedirs(results_dir, exist_ok=True)

reg0 = RandomForestRegressor(random_state=0, n_jobs=8)
reg1 = RandomForestRegressor(random_state=0, n_jobs=8)
reg2 = RandomForestRegressor(random_state=0, n_jobs=8)
reg3 = RandomForestRegressor(random_state=0, n_jobs=8)

with open(os.path.join(results_dir, "progress.log"), "a", buffering=1) as log, contextlib.redirect_stdout(log):
    print(f"\n##### Run started {time.ctime()} #####")

    if(RSCV==False):
        rf_tas = reg0.fit(X_train,y_train_tas)
        rf_pr = reg1.fit(X_train,y_train_pr)
        rf_pr90 = reg2.fit(X_train,y_train_pr90)
        rf_dtr = reg3.fit(X_train,y_train_dtr)
    else:
        rf_random0 = RandomizedSearchCV(estimator = reg0, param_distributions = random_grid, n_iter = 29, cv = 3, verbose=2, n_jobs = 1)
        rf_random1 = RandomizedSearchCV(estimator = reg1, param_distributions = random_grid, n_iter = 29, cv = 3, verbose=2, n_jobs = 1)
        rf_random2 = RandomizedSearchCV(estimator = reg2, param_distributions = random_grid, n_iter = 29, cv = 3, verbose=2, n_jobs = 1)
        rf_random3 = RandomizedSearchCV(estimator = reg3, param_distributions = random_grid, n_iter = 29, cv = 3, verbose=2, n_jobs = 1)

        print(f"=== tas started {time.ctime()} ===")
        rf_tas = rf_random0.fit(X_train,y_train_tas)
        json.dump(rf_tas.best_params_, open(os.path.join(results_dir, "best_params_tas.json"), "w"))

        print(f"=== pr started {time.ctime()} ===")
        rf_pr = rf_random1.fit(X_train,y_train_pr)
        json.dump(rf_pr.best_params_, open(os.path.join(results_dir, "best_params_pr.json"), "w"))

        print(f"=== pr90 started {time.ctime()} ===")
        rf_pr90 = rf_random2.fit(X_train,y_train_pr90)
        json.dump(rf_pr90.best_params_, open(os.path.join(results_dir, "best_params_pr90.json"), "w"))

        print(f"=== dtr started {time.ctime()} ===")
        rf_dtr = rf_random3.fit(X_train,y_train_dtr)
        json.dump(rf_dtr.best_params_, open(os.path.join(results_dir, "best_params_dtr.json"), "w"))

    print(f"##### All done {time.ctime()} #####")

print('done!')

done!


In [18]:
if(RSCV==True):
    print(rf_tas.best_params_)
    print(rf_pr.best_params_)
    print(rf_pr90.best_params_)
    print(rf_dtr.best_params_)

{'n_estimators': 250, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 1.0, 'max_depth': 50, 'bootstrap': True}
{'n_estimators': 250, 'min_samples_split': 25, 'min_samples_leaf': 12, 'max_features': 1.0, 'max_depth': 45, 'bootstrap': True}
{'n_estimators': 250, 'min_samples_split': 10, 'min_samples_leaf': 8, 'max_features': 1.0, 'max_depth': 45, 'bootstrap': True}
{'n_estimators': 150, 'min_samples_split': 5, 'min_samples_leaf': 8, 'max_features': 'sqrt', 'max_depth': None, 'bootstrap': True}


**load test set ssp245**

In [19]:
test_Y = xr.open_dataset(data_path + 'outputs_ssp245.nc').compute()
test_X = xr.open_dataset(data_path + 'inputs_ssp245.nc').compute()

tas_truth = test_Y["tas"].mean('member')
pr_truth = test_Y["pr"].mean('member') * SECONDS_PER_DAY
pr90_truth = test_Y["pr90"].mean('member') * SECONDS_PER_DAY
dtr_truth = test_Y["diurnal_temperature_range"].mean('member')

test_inputs = pd.DataFrame({
    "CO2": normalize_co2(test_X["CO2"].data),
    "CH4": normalize_ch4(test_X["CH4"].data)
}, index=test_X["CO2"].coords['time'].data)

### Combine with aerosol EOFs
test_inputs=pd.concat([test_inputs, 
                       bc_solver.projectField(test_X["BC"], neofs=5, eofscaling=1).to_dataframe().unstack('mode').rename(columns={i:f"BC_{i}" for i in range(5)}),
                       so2_solver.projectField(test_X["SO2"], neofs=5, eofscaling=1).to_dataframe().unstack('mode').rename(columns={i:f"_{i}" for i in range(5)}),
                       ], axis=1)

In [20]:
m_out_t   = rf_tas.predict(test_inputs.values)
m_out_p   = rf_pr.predict(test_inputs.values)
m_out_p90 = rf_pr90.predict(test_inputs.values)
m_out_d   = rf_dtr.predict(test_inputs.values)

m_out_tas  = m_out_t.reshape(86, 96, 144)
m_out_pr   = m_out_p.reshape(86, 96, 144)
m_out_pr90 = m_out_p90.reshape(86, 96, 144)
m_out_dtr  = m_out_d.reshape(86, 96, 144)

In [21]:
xr_output=xr.Dataset(coords={'time': test_X.time.values, 'lat': test_X.latitude.values, 'lon': test_X.longitude.values})
xr_output["tas"]=(['time', 'lat', 'lon'],  m_out_tas)
xr_output["diurnal_temperature_range"]=(['time', 'lat', 'lon'],  m_out_dtr)
xr_output["pr"]=(['time', 'lat', 'lon'],  m_out_pr)
xr_output["pr90"]=(['time', 'lat', 'lon'],  m_out_pr90)
    
#save output to netcdf 
xr_output.to_netcdf(path_output,'w')

In [22]:
print(f"RMSE: {get_rmse(tas_truth[35], m_out_tas[35])}")
print(f"RMSE: {get_rmse(tas_truth[85], m_out_tas[85])}")
print(f"RMSE: {get_rmse(tas_truth[30:40], m_out_tas[30:40])}")
print(f"RMSE: {get_rmse(tas_truth[75:], m_out_tas[75:])}")
print(f"RMSE: {get_rmse(tas_truth[35:], m_out_tas[35:])}")
print("\n")

print(f"RMSE: {get_rmse(dtr_truth[35], m_out_dtr[35])}")
print(f"RMSE: {get_rmse(dtr_truth[85], m_out_dtr[85])}")
print(f"RMSE: {get_rmse(dtr_truth[30:40], m_out_dtr[30:40])}")
print(f"RMSE: {get_rmse(dtr_truth[75:], m_out_dtr[75:])}")
print(f"RMSE: {get_rmse(dtr_truth[35:], m_out_dtr[35:])}")
print("\n")

print(f"RMSE: {get_rmse(pr_truth[35], m_out_pr[35])}")
print(f"RMSE: {get_rmse(pr_truth[85], m_out_pr[85])}")
print(f"RMSE: {get_rmse(pr_truth[30:40], m_out_pr[30:40])}")
print(f"RMSE: {get_rmse(pr_truth[75:], m_out_pr[75:])}")
print(f"RMSE: {get_rmse(pr_truth[35:], m_out_pr[35:])}")
print("\n")

print(f"RMSE: {get_rmse(pr90_truth[35], m_out_pr90[35])}")
print(f"RMSE: {get_rmse(pr90_truth[85], m_out_pr90[85])}")
print(f"RMSE: {get_rmse(pr90_truth[30:40], m_out_pr90[30:40])}")
print(f"RMSE: {get_rmse(pr90_truth[75:], m_out_pr90[75:])}")
print(f"RMSE: {get_rmse(pr90_truth[35:], m_out_pr90[35:])}")

RMSE: 0.3169199040989382
RMSE: 0.42840330265258586
RMSE: 0.37804014782787243
RMSE: 0.44050453366931597
RMSE: 0.5303087422112003


RMSE: 0.1365951885771623
RMSE: 0.17152466996976284
RMSE: 0.14355932158755913
RMSE: 0.20114214353365042
RMSE: 0.18430869381350387


RMSE: 0.40760379867432667
RMSE: 0.6104551140788216
RMSE: 0.5010036866950871
RMSE: 0.540975655772639
RMSE: 0.5372637326417918


RMSE: 1.3036213016273017
RMSE: 1.724099861113039
RMSE: 1.4868582751415467
RMSE: 1.5511314147929398
RMSE: 1.5624175997311307


In [23]:
#predict avg last 20 years

m_out_t = rf_tas.predict(test_inputs[-20:].mean().values.reshape(1,12))
m_out_p = rf_pr.predict(test_inputs[-20:].mean().values.reshape(1,12))
m_out_p90 = rf_pr90.predict(test_inputs[-20:].mean().values.reshape(1,12))
m_out_d = rf_dtr.predict(test_inputs[-20:].mean().values.reshape(1,12))

m_out_tas = m_out_t.reshape(96, 144)
m_out_pr = m_out_p.reshape(96, 144)
m_out_pr90 = m_out_p90.reshape(96, 144)
m_out_dtr = m_out_d.reshape(96, 144)

In [24]:
tas_truth2=tas_truth[-20:].mean(dim="time")
pr_truth2=pr_truth[-20:].mean(dim="time")
pr90_truth2=pr90_truth[-20:].mean(dim="time")
dtr_truth2=dtr_truth[-20:].mean(dim="time")

print(f"RMSE: {get_rmse(tas_truth2, m_out_tas)}")
print(f"RMSE: {get_rmse(dtr_truth2, m_out_dtr)}")
print(f"RMSE: {get_rmse(pr_truth2, m_out_pr)}")
print(f"RMSE: {get_rmse(pr90_truth2, m_out_pr90)}")

RMSE: 0.23667236535954622
RMSE: 0.16779578175299287
RMSE: 0.2063767850521281
RMSE: 0.5686329957185584
